In [12]:
import pandas as pd 
import numpy as np
import torch
from torch_geometric.data import Data
import random

In [13]:
ppi = pd.read_csv("../data/processed/kidney_PPI_final.csv")
ppi


,A,B,combined_score,coexpression
0,MAP2K4,FLNC,2,0.285217
1,FNTA,ACVR1,2,0.014917
2,GATA2,PML,2,-0.098040
3,RPA2,STAT3,2,0.350444
4,ARF1,GGA3,2,0.304046
...,...,...,...,...
1528071,LAMA2,FBLN2,1,0.301295
1528072,BMP1,DSPP,1,-0.197264
1528073,SNAI1,LOXL1,1,0.159567
1528074,FBLN1,COL18A1,2,0.070642


In [14]:
ppi_filtered = ppi[ppi['coexpression'].abs() >= 0.2]


In [15]:
labels = pd.read_csv ("../data/Final/labels.csv")

In [16]:
expressions = pd.read_csv("../data/Final/expressions.csv")

In [17]:
mutations = pd.read_csv("../data/Final/mutations.csv")
mutations

,ModelID,SAMD11,NOC2L,KLHL17,PLEKHN1,PERM1,HES4,ISG15,AGRN,C1orf159,...,NLGN4Y,KDM5D,PRORY,IFT25,PCBD1,CEBPZOS,VPREB3,SCOC,SMIM3,LSM5
0,ACH-001398,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ACH-000250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ACH-000684,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ACH-000792,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ACH-000907,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,ACH-000649,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,ACH-000313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,ACH-000385,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,ACH-000096,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,ACH-002533,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
ppi.shape


(1528076, 4)

In [19]:
labels.shape


(32, 17864)

In [20]:
expressions.shape


(32, 18360)

In [21]:
mutations.shape

(32, 17150)

In [22]:
ppi_genes = pd.concat([ppi_filtered['A'], ppi_filtered['B']]).unique()
gene_to_idx = {gene: i for i, gene in enumerate(ppi_genes)}



In [23]:
len(gene_to_idx)

17306

In [24]:
edge_src = ppi_filtered['A'].map(gene_to_idx).values
edge_dst = ppi_filtered["B"].map(gene_to_idx).values

In [25]:
edge_src[:5]

array([0, 1, 2, 3, 4])

In [26]:
edge_index = torch.tensor([edge_src,edge_dst],dtype=torch.long)
edge_index.shape

/tmp/ipykernel_5813/3632093261.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  edge_index = torch.tensor([edge_src,edge_dst],dtype=torch.long)


torch.Size([2, 735802])

In [27]:
edge_attr = torch.tensor(ppi_filtered[['combined_score', 'coexpression']].values, dtype=torch.float)
edge_attr = (edge_attr - edge_attr.mean(dim=0)) / edge_attr.std(dim=0)
edge_attr.shape

torch.Size([735802, 2])

In [28]:
ppi_genes[:5]           # gene_to_idx order
  # expression column order


<StringArray>
['MAP2K4', 'RPA2', 'ARF1', 'ARFIP2', 'ARFIP1']
Length: 5, dtype: str

In [29]:
expressions.columns[1:6]


Index(['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'FIRRM'], dtype='str')

In [30]:
mut_no_id = mutations.drop('ModelID', axis=1)
mutations_aligned = mut_no_id.reindex(columns=ppi_genes, fill_value=0)


In [47]:
expr_no_id = expressions.drop('ModelID', axis=1)
expr_data = expr_no_id[ppi_genes]
std = expr_data.std(axis=0)
std[std == 0] = 1
expr_data = (expr_data - expr_data.mean(axis=0)) / std



In [48]:
labels_no_id = labels.drop('ModelID', axis=1)


In [49]:
labels_reindexed = labels_no_id.reindex(columns=ppi_genes)

graph_list = []
for i in range(32):
    expr_row = expr_data.iloc[i].values
    mut_row = mutations_aligned.iloc[i].values
    x = torch.tensor(np.column_stack([expr_row, mut_row]), dtype=torch.float)
    
    y_raw = labels_reindexed.iloc[i]
    cell_mask = torch.tensor((~y_raw.isna().values) & np.isin(ppi_genes, labels_no_id.columns), dtype=torch.bool)
    y = torch.tensor(y_raw.fillna(0).values, dtype=torch.float)
    
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, mask=cell_mask)
    graph_list.append(data)



In [50]:
len(graph_list)

32

In [51]:
graph_list

[Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[17306], mask=[17306]),
 Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[17306], mask=[17306]),
 Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[17306], mask=[17306]),
 Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[17306], mask=[17306]),
 Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[17306], mask=[17306]),
 Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[17306], mask=[17306]),
 Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[17306], mask=[17306]),
 Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[17306], mask=[17306]),
 Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[17306], mask=[17306]),
 Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[17306], mask=[17306]),
 Data(x=[17306, 2], edge_index=[2, 735802], edge_attr=[735802, 2], y=[

In [52]:
random.seed(12)
random.shuffle(graph_list)

In [53]:
train_data = graph_list[:26]
test_data = graph_list[26:]

In [54]:
len(train_data)


26

In [55]:
from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1)



In [56]:
len(graph_list)




32

In [57]:
import torch.nn as nn
from torch_geometric.nn import GATv2Conv

class KidneyGAT(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GATv2Conv(2, 64, heads=2, edge_dim=2)
        self.conv2 = GATv2Conv(64 * 2, 64, heads=2, edge_dim=2, concat=False)
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(64, 1)
    
    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        
        x = self.conv1(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.conv2(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.out(x)
        return x.squeeze()




In [58]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = KidneyGAT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
loss_fn = nn.MSELoss()

In [59]:
torch.cuda.empty_cache()


In [60]:
expr_data.isna().sum().sum()


np.int64(0)

In [61]:
epochs = 200

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        
        pred = model(batch)
        loss = loss_fn(pred[batch.mask], batch.y[batch.mask])
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


Epoch 10, Loss: 0.1910
Epoch 20, Loss: 0.1769
Epoch 30, Loss: 0.1712
Epoch 40, Loss: 0.1688
Epoch 50, Loss: 0.1676
Epoch 60, Loss: 0.1660
Epoch 70, Loss: 0.1655
Epoch 80, Loss: 0.1648
Epoch 90, Loss: 0.1641
Epoch 100, Loss: 0.1636
Epoch 110, Loss: 0.1643
Epoch 120, Loss: 0.1633
Epoch 130, Loss: 0.1626
Epoch 140, Loss: 0.1634
Epoch 150, Loss: 0.1615
Epoch 160, Loss: 0.1613
Epoch 170, Loss: 0.1613
Epoch 180, Loss: 0.1613
Epoch 190, Loss: 0.1604
Epoch 200, Loss: 0.1603


In [62]:
model.eval()
test_losses = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = model(batch)
        loss = loss_fn(pred[batch.mask], batch.y[batch.mask])
        test_losses.append(loss.item())

print(f"Test MSE: {np.mean(test_losses):.4f}")



Test MSE: 0.1915


In [63]:
from scipy.stats import pearsonr

correlations = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = model(batch)
        p = pred[batch.mask].cpu().numpy()
        a = batch.y[batch.mask].cpu().numpy()
        r, _ = pearsonr(p, a)
        correlations.append(r)

print(f"Test Pearson r: {np.mean(correlations):.4f}")


Test Pearson r: 0.3640
